In [2]:
import nltk
import json
import umap
import os

import pandas as pd
import numpy as np
import seaborn as sns
import plotly.express as px
import torch
import matplotlib.pyplot as plt
import hashlib
import plotly.graph_objects as go

from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from gensim.models import Word2Vec
from transformers import AutoTokenizer, AutoModel
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from functools import lru_cache
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from storage.database import Database

c:\Users\User\Desktop\Micael\Furb\PLN\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
DATABASE_PATH = "../data.db"
QUANTITY = 1000
CAMPOS_ID =("id", "id_discurso")
SEED = 42
database = Database(DATABASE_PATH)
query = "educacao publica investimento escolas professores"

modelos = {
    "mBERT": "bert-base-multilingual-cased",
    "BERTimbau": "neuralmind/bert-base-portuguese-cased",
    "BERTugues": "ricardoz/BERTugues-base-portuguese-cased"
}

try:
    stopwords.words('portuguese')
except LookupError:
    nltk.download('stopwords')
    
nltk.download('rslp')
nltk.download('punkt_tab')

[nltk_data] Downloading package rslp to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package rslp is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Métodos

In [4]:
def mostrar_tabela(x,vec,title):
    df_bow = pd.DataFrame(x.toarray(), columns=vec.get_feature_names_out())
    print(title)
    display(df_bow)

In [5]:
def parse_valor(valor):
    if not valor:
        return None

    # Se já for lista/dict, retorna direto
    if isinstance(valor, (list, dict)):
        return valor

    # Se for string, tenta converter
    if isinstance(valor, str):
        valor = valor.strip()

        if not valor:
            return None

        try:
            return json.loads(valor)
        except json.JSONDecodeError:
            return valor  # texto normal

    return valor

In [6]:
def get_corpus(documentos, campo, campos_id=("id", "id_doc")):
    corpus = []
    doc_ids = []

    for d in documentos:
        if campo in d and d[campo]:
            valor = d[campo]
            j = parse_valor(valor)
            corpus.append(j)
            
            id_composto = "_".join(str(d[c]) for c in campos_id)
            doc_ids.append(id_composto)

    return corpus, doc_ids

In [7]:
def get_discursos_corpus(campo, campos_id, size):
    discursos = database.get_discursos(size)
    return get_corpus(discursos, campo, campos_id)

In [8]:
def processar_dados_bow(corpus):    
    bow_vec = CountVectorizer(
        lowercase=False,
        preprocessor=None,
        tokenizer=lambda x: x
    )
    
    X_bow = bow_vec.fit_transform(corpus)
    
    return X_bow, bow_vec

In [9]:
def processar_dados_td_idf(corpus):
    tfidf_vec = TfidfVectorizer(
        lowercase=False,
        preprocessor=None,
        tokenizer=lambda x: x
    )
    
    X_tfidf = tfidf_vec.fit_transform(corpus)
    
    return X_tfidf, tfidf_vec

In [10]:
def search_and_rank(query, vectorizer, X_corpus, corpus, doc_ids, method_name):
    """
    Vetoriza uma query, calcula a similaridade com o corpus e exibe os resultados.
    """
    q_vec = vectorizer.transform([query])
    sim_scores = cosine_similarity(q_vec, X_corpus).ravel()
    rank = np.argsort(sim_scores)[::-1]

    print(f"Top-3 Similares para a Query (usando {method_name}):")
    for i in rank[:3]:
        if sim_scores[i] > 0.01: # Apenas mostra se houver alguma similaridade
            print(f"  Doc{doc_ids[i]} (score={sim_scores[i]:.3f}): {corpus[i]}")
    print("-" * 40)

In [11]:
def matriz_similaridade_top_n(dados, doc_ids, top_n, subtitle = ""):
    sim_matrix = cosine_similarity(dados, dados)

    sim_no_diag = sim_matrix.copy()
    np.fill_diagonal(sim_no_diag, 0)

    mean_sim = sim_no_diag.mean(axis=1)
    top_idxs = np.argsort(mean_sim)[::-1][:top_n]

    sub_sim = sim_matrix[np.ix_(top_idxs, top_idxs)]
    
    if doc_ids is None:
        doc_ids = np.arange(len(dados))

    doc_labels = np.array(doc_ids)[top_idxs]
    df_similarity = pd.DataFrame(sub_sim, index=doc_labels, columns=doc_labels)

    plt.figure(figsize=(8, 6))
    sns.heatmap(df_similarity, annot=True, cmap="Blues", fmt=".2f")
    plt.title(f"Heatmap dos {top_n} documentos mais semelhantes - {subtitle}")
    plt.xlabel("Documentos")
    plt.ylabel("Documentos")
    plt.tight_layout()
    plt.show()

In [12]:
def grafico2D(dados):
    pca = PCA(n_components=2)
    X_tfidf_pca = pca.fit_transform(dados.toarray())

    plt.figure(figsize=(10, 7))
    plt.scatter(X_tfidf_pca[:, 0], X_tfidf_pca[:, 1], c='blue', alpha=0.7, s=100)
    plt.title("Visualização 2D dos Vetores de Documentos")
    plt.xlabel("Componente Principal 1")
    plt.ylabel("Componente Principal 2")
    plt.grid(True)

    plt.show()

In [13]:
def grafico3D(dados):
    # Redução de dimensionalidade
    pca = PCA(n_components=3)
    X_tfidf_pca = pca.fit_transform(dados.toarray())

    # Criando gráfico 3D com Plotly Express
    fig = px.scatter_3d(
        x=X_tfidf_pca[:, 0],
        y=X_tfidf_pca[:, 1],
        z=X_tfidf_pca[:, 2],
        opacity=0.7,
        title="Visualização 3D dos Vetores de Documentos"
    )

    # Ajustando labels
    fig.update_layout(
        scene=dict(
            xaxis_title="Componente Principal 1",
            yaxis_title="Componente Principal 2",
            zaxis_title="Componente Principal 3"
        )
    )

    fig.show()

In [14]:
def calcula_vetor_medio(corpus, modelo, vector_dim):
    vetores_documentos = []
    
    for doc_tokens in corpus:
        # Pega os vetores das palavras no documento que existem no modelo
        vetores_palavras = [modelo.wv[palavra] for palavra in doc_tokens if palavra in modelo.wv]

        if len(vetores_palavras) > 0:
            # Calcula a média dos vetores das palavras
            vetor_medio_doc = np.mean(vetores_palavras, axis=0)
            vetores_documentos.append(vetor_medio_doc)
        else:
            # Se nenhuma palavra do documento estiver no vocabulário, adiciona um vetor de zeros
            # (Isso é raro com min_count=1, mas é bom ter por segurança)
            print(f"Aviso: Documento '{' '.join(doc_tokens)}' não possui palavras no vocabulário do modelo.")
            vetores_documentos.append(np.zeros(vector_dim))

    # Garantir que temos um array NumPy
    return np.array(vetores_documentos)

In [15]:
def recomendar_documentos_w2v(consulta, model_w2v, documentos_originais, vetores_documentos, num_resultados=None):
    """
    Recomenda documentos com base na similaridade semântica (Word2Vec) com a consulta.

    Args:
        consulta (str): O termo ou frase de busca do usuário.
        model_w2v (gensim.models.Word2Vec): O modelo Word2Vec treinado.
        documentos_originais (list): Lista das strings dos documentos originais.
        vetores_documentos (np.ndarray): Array NumPy com os vetores médios pré-calculados
                                         para cada documento original.
        num_resultados (int, optional): Número máximo de resultados a retornar.
                                         Se None, retorna todos. Defaults to None.

    Returns:
        pd.DataFrame: DataFrame com colunas 'Documento' e 'Similaridade',
                      ordenado por similaridade decrescente. Retorna DataFrame vazio
                      se a consulta não puder ser vetorizada (nenhuma palavra conhecida).
    """
    # 1. Preprocessar (tokenizar, lowercase) a consulta
    tokens_busca = word_tokenize(consulta.lower())
    
    if not tokens_busca:
        print("Aviso: Consulta vazia após tokenização.")
        return pd.DataFrame({'Documento': [], 'Similaridade': []})

    # 2. Vetorizar a consulta usando o modelo Word2Vec (média dos vetores)
    vetores_palavras_busca = [model_w2v.wv[token] for token in tokens_busca if token in model_w2v.wv]

    # Verifica se alguma palavra da consulta foi encontrada no vocabulário
    if not vetores_palavras_busca:
        print(f"Aviso: Nenhuma palavra da consulta '{consulta}' encontrada no vocabulário do modelo.")
        # Retorna DataFrame vazio se a consulta não tem palavras conhecidas
        return pd.DataFrame({'Documento': [], 'Similaridade': []})

    # Calcula o vetor médio para a busca
    vetor_busca = np.mean(vetores_palavras_busca, axis=0)

    # 3. Calcular a similaridade por cosseno entre a consulta e os documentos
    # O vetor_busca precisa ser 2D (1, N_dims) para a função cosine_similarity
    similaridades = cosine_similarity(vetor_busca.reshape(1, -1), vetores_documentos)[0] # Pega a primeira (e única) linha de scores

    # 4. Obter os índices dos documentos ordenados pela similaridade (decrescente)
    indices_ranqueados = np.argsort(similaridades)[::-1]

    # 5. Criar o DataFrame de resultados com base nos índices ranqueados
    # Seleciona os documentos originais e as similaridades na ordem correta
    docs_ranqueados = [documentos_originais[i] for i in indices_ranqueados]
    scores_ranqueados = similaridades[indices_ranqueados]

    recomendacoes = pd.DataFrame({
        'Documento': docs_ranqueados,
        'Similaridade': scores_ranqueados
    })

    # 6. Limitar o número de resultados, se especificado
    if num_resultados is not None:
        recomendacoes = recomendacoes.head(num_resultados)

    return recomendacoes

In [16]:
def mostrar_grafico_clusters(vetores_documentos,df_w2v_docs):
    # 3. Redução de Dimensionalidade com PCA para Visualização
    # PCA para reduzir para 3 componentes para visualização 3D
    pca_w2v = PCA(n_components=3)

    # Aplicando PCA aos vetores Word2Vec (não precisa de .toarray())
    caracteristicas_reduzidas_w2v = pca_w2v.fit_transform(vetores_documentos)

    # 4. Preparar DataFrame para Plotagem
    # Criando um DataFrame que inclui as 3 componentes PCA, Cluster e Documento
    pca_w2v_df = pd.DataFrame(
        caracteristicas_reduzidas_w2v,
        columns=['Componente PCA 1', 'Componente PCA 2', 'Componente PCA 3']
    )
    # Adiciona as informações de cluster e documento ao DataFrame do PCA
    pca_w2v_df['Cluster'] = df_w2v_docs['Cluster']
    pca_w2v_df['Documento Original'] = df_w2v_docs['Documento Original']

    # Convertendo os rótulos dos clusters para string para cores categóricas no Plotly
    pca_w2v_df['Cluster'] = pca_w2v_df['Cluster'].astype(str)

    # 5. Criar Gráfico 3D Interativo com Plotly Express
    fig_w2v = px.scatter_3d(
        pca_w2v_df,
        x='Componente PCA 1',
        y='Componente PCA 2',
        z='Componente PCA 3',        # Mapeando a terceira componente para o eixo Z
        color='Cluster',             # Colorindo os pontos pelo cluster atribuído
        title='Clusters de Documentos (Word2Vec + PCA 3D)', # Título atualizado
        labels={                     # Rótulos dos eixos
            'Componente PCA 1': 'Componente PCA 1',
            'Componente PCA 2': 'Componente PCA 2',
            'Componente PCA 3': 'Componente PCA 3'
        },
        width=900,                   # Largura/Altura do gráfico (ajustável)
        height=700,
        hover_data={                 # Dados a exibir ao passar o mouse sobre um ponto
            'Componente PCA 1': ':.2f', # Formata PCA para 2 casas decimais no hover
            'Componente PCA 2': ':.2f',
            'Componente PCA 3': ':.2f',
            'Cluster': True, # Mostra o número do cluster
            'Documento Original': True # Mostra o texto do documento original
        }
        # Alternativa para hover_data: hover_data=['Documento Original', 'Cluster']
    )

    # Opcional: Ajustar a aparência dos marcadores
    # fig_w2v.update_traces(marker=dict(size=5, opacity=0.8))

    # Exibir o gráfico interativo
    fig_w2v.show()

In [17]:
def k_means(documento_original, vetores_documentos, num_clusters):
    # 1. Preparar DataFrame Inicial (se ainda não existir)
    df_w2v_docs = pd.DataFrame({'Documento Original': documento_original})

    # Aplicando o KMeans aos vetores Word2Vec
    # Adicionar n_init=10 para evitar warnings e melhorar a robustez
    kmeans_w2v = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)

    # O fit é feito diretamente nos vetores Word2Vec (que já são densos)
    kmeans_w2v.fit(vetores_documentos)

    # Adicionando as labels (rótulos) dos clusters ao DataFrame
    df_w2v_docs['Cluster'] = kmeans_w2v.labels_

    return df_w2v_docs

In [18]:
def busca_termos_similares(modelo, documentos_originais, vetores_documentos,num_resultados):
    # Entrada pelo usuário
    termo_busca_usuario = input("Digite os termos de busca para Word2Vec: ")

    # Obter as recomendações usando a função
    # Você pode definir um número máximo de resultados aqui, se quiser (ex: num_resultados=5)
    recomendacoes_df = recomendar_documentos_w2v(
        termo_busca_usuario,
        modelo,
        documentos_originais,
        vetores_documentos,
        num_resultados
    )

    print(f"termo pesquisado: {termo_busca_usuario}")

    # Exibir o DataFrame resultante
    if not recomendacoes_df.empty:
        print("Ok")
    else:
        print("Nenhuma recomendação encontrada para o termo de busca.")

    return recomendacoes_df

In [19]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


@lru_cache(maxsize=None)
def load_model(model_name, device):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()
    return tokenizer, model


@torch.no_grad()
def encode_sentences_dual(model_name, sentences, batch_size=32):
    device = DEVICE
    tokenizer, model = load_model(model_name, device)
    
    print("Device:", DEVICE)

    all_cls = []
    all_mean = []

    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(device)

        if device == "cuda":
            with torch.cuda.amp.autocast():
                outputs = model(**inputs)
        else:
            outputs = model(**inputs)

        hidden = outputs.last_hidden_state

        cls_vec = hidden[:, 0, :]
        mean_vec = mean_pooling(hidden, inputs["attention_mask"])

        all_cls.append(cls_vec.cpu().numpy())
        all_mean.append(mean_vec.cpu().numpy())

    return np.vstack(all_cls), np.vstack(all_mean)


def hash_all(documentos, modelos):
    base = "".join(documentos) + str(modelos)
    return hashlib.md5(base.encode()).hexdigest()

def salvar_embeds(path, embeds):
    np.savez_compressed(
        path,
        **{f"{k[0]}__{k[1]}": v for k, v in embeds.items()}
    )

def carregar_embeds(path):
    data = np.load(path)
    embeds = {}
    for key in data.files:
        nome, pooling = key.split("__")
        embeds[(nome, pooling)] = data[key]
    return embeds

def gerar_embeds(documentos, force_recompute=False):
    hash_id = hash_all(documentos, modelos)
    path = f"embeds_{hash_id}.npz"

    if os.path.exists(path) and not force_recompute:
        return carregar_embeds(path)

    embeds = {}

    for nome_visivel, nome_hf in modelos.items():
        X_cls, X_mean = encode_sentences_dual(nome_hf, documentos)
        embeds[(nome_visivel, "CLS")] = X_cls
        embeds[(nome_visivel, "MEAN")] = X_mean

    salvar_embeds(path, embeds)

    print({k: v.shape for k, v in embeds.items()})
    return embeds

In [20]:
def plot_similarity_heatmap_px(
    X,
    sentences=None,          # lista de sentenças para o hover
    title="Similaridade (cosseno)",
    cmap="Blues",
    mask_upper=False,        # True: mostra só triângulo inferior
    vmin=0.0, vmax=1.0,
    cbar_label="Similaridade",
    width=700, height=600,
    fmt=".2f",               # formatação do label interno
    text_font_size=11,
    text_font_color="black",
    xgap=1, ygap=1           # “espessura” das linhas entre células
):
    # 1) Similaridade
    S = cosine_similarity(X)
    n = S.shape[0]
    labels = [f"S{i+1}" for i in range(n)]

    # 2) Máscara (triângulo superior)
    Z = S.astype(float).copy()
    if mask_upper:
        iu = np.triu_indices(n, k=1)
        Z[iu] = np.nan

    # 3) Labels numéricos dentro das células (vão como text)
    text_matrix = np.empty((n, n), dtype=object)
    text_matrix[:] = ""
    for i in range(n):
        for j in range(n):
            if not np.isnan(Z[i, j]):
                text_matrix[i, j] = f"{Z[i, j]:{fmt}}"

    # 4) Hover somente com as sentenças (sem número)
    #    Usamos customdata com HTML simples para melhor legibilidade
    customdata = np.empty((n, n), dtype=object)
    customdata[:] = ""
    for i in range(n):
        for j in range(n):
            if not np.isnan(Z[i, j]):
                s1 = sentences[i] if sentences is not None else labels[i]
                s2 = sentences[j] if sentences is not None else labels[j]
                customdata[i, j] = f"<b>{labels[i]}</b>: {s1}<br><b>{labels[j]}</b>: {s2}"

    # 5) Construir heatmap
    fig = go.Figure(
        data=go.Heatmap(
            z=Z,
            x=labels,
            y=labels,
            zmin=vmin, zmax=vmax,
            colorscale=cmap,
            colorbar=dict(title=cbar_label),
            # labels internos
            text=text_matrix,
            texttemplate="%{text}",
            textfont=dict(color=text_font_color, size=text_font_size),
            # hover apenas com as frases
            customdata=customdata,
            hovertemplate="%{customdata}<extra></extra>",
            # “grades” entre células
            xgap=xgap, ygap=ygap
        )
    )

    # Layout
    fig.update_layout(
        title=title,
        width=width, height=height,
        template="plotly_white",
        margin=dict(l=60, r=30, t=60, b=60),
    )
    # Células quadradas e origem no topo
    fig.update_yaxes(autorange="reversed", scaleanchor="x", scaleratio=1)

    fig.show()
    return S, fig


def run_kmeans(X, n_clusters=3, random_state=SEED):
    km = KMeans(n_clusters=n_clusters, random_state=random_state, n_init='auto')
    labels = km.fit_predict(X)
    return labels

def reduce_2d(X, method="pca", random_state=SEED, n_neighbors=15, min_dist=0.1):
    method = method.lower()
    if method == "pca":
        reducer = PCA(n_components=2, random_state=random_state)
        Z = reducer.fit_transform(X)
        meta = ("PCA", reducer.explained_variance_ratio_)
        return Z, meta
    elif method == "umap":
        reducer = umap.UMAP(
            n_components=2,
            random_state=random_state,
            n_neighbors=n_neighbors,
            min_dist=min_dist,
            metric="euclidean",
        )
        Z = reducer.fit_transform(X)
        meta = ("UMAP", None)
        return Z, meta
    else:
        raise ValueError("method deve ser 'pca' ou 'umap'")

def plot_scatter_embeddings(embeds, sentences, method="pca", random_state=SEED):
    """
    Gera um DataFrame 2D concatenando projeções por (modelo, pooling).
    embeds: dict[(modelo, pooling)] -> np.ndarray [N, H]
    sentences: lista de sentenças (mesmo N, ordem consistente)
    """
    rows = []
    for (modelo, pooling), X in embeds.items():
        Z, meta = reduce_2d(X, method=method, random_state=random_state)
        # Z: [N,2]
        df_tmp = pd.DataFrame({
            "x": Z[:, 0],
            "y": Z[:, 1],
            "modelo": modelo,
            "pooling": pooling,
            "sentenca": sentences,   # mantém coerência com clusters_df
        })
        rows.append(df_tmp)

    df2d = pd.concat(rows, ignore_index=True)
    return df2d

## Testes

Bow (Bag-of-Words)

In [ ]:
bow_corpus_stem, doc_ids_bow_stem = get_discursos_corpus("transcricao_stemizado", CAMPOS_ID, QUANTITY)
x_bow_stem, bow_vec_stem = processar_dados_bow(bow_corpus_stem)

### Transcrição

In [ ]:
corpus_transcricao,doc_ids_transcricao = get_discursos_corpus("transcricao", CAMPOS_ID, QUANTITY)

corpus_tf_idf_token,doc_ids_tf_idf_token = get_discursos_corpus("transcricao_tokens",CAMPOS_ID, QUANTITY)
x_tf_idf_token,vec_tf_idf_token = processar_dados_td_idf(corpus_tf_idf_token)

corpus_tf_idf_stem, doc_ids_tf_idf_stem = get_discursos_corpus("transcricao_stemizado",CAMPOS_ID, QUANTITY)
x_tf_idf_stem,vec_tf_idf_stem = processar_dados_td_idf(corpus_tf_idf_stem)

corpus_tf_idf_lem, doc_ids_tf_ids_lem = get_discursos_corpus("transcricao_lemizado",CAMPOS_ID, QUANTITY)
x_tf_idf_lem,vec_tf_idf_lem = processar_dados_td_idf(corpus_tf_idf_lem)

In [ ]:
mostrar_tabela(x_tf_idf_token,vec_tf_idf_token,"Matriz Documento-Termo Transcrição (TF-IDF)")

In [ ]:
print(f">> Executando busca para a query: '{query}'\n")

search_and_rank(query, bow_vec_stem, x_bow_stem, bow_corpus_stem, doc_ids_bow_stem, "BoW")

search_and_rank(query, vec_tf_idf_token, x_tf_idf_token, corpus_tf_idf_token, doc_ids_tf_idf_token, "TF-IDF - token")
search_and_rank(query, vec_tf_idf_stem, x_tf_idf_stem, corpus_tf_idf_stem, doc_ids_tf_idf_stem, "TF-IDF - stem")
search_and_rank(query, vec_tf_idf_lem, x_tf_idf_lem, corpus_tf_idf_lem, doc_ids_tf_ids_lem, "TF-IDF - lem")

In [ ]:
matriz_similaridade_top_n(x_tf_idf_token, doc_ids_tf_idf_token,15, "transcrição - token")

In [ ]:
discurso_a = database.get_discurso_by_id(204497,12)
discurso_b = database.get_discurso_by_id(204497, 14)

print(discurso_a["transcricao"])
print('-'*50)
print(discurso_b["transcricao"])

In [ ]:
grafico2D(x_tf_idf_token)

In [ ]:
grafico3D(x_tf_idf_token)

##### Word2Vec

In [ ]:
vector_dim = 200 # Dimensão dos vetores
window_size = 8  # Janela de contexto
min_word_count = 3 # Contagem mínima para incluir palavra
num_workers = 4 # Threads para treinamento
sg = 1 # Usando CBOW (padrão), 1 para Skip-gram
num_clusters = 10

model_w2v_transcricao_token = Word2Vec(
    sentences=corpus_tf_idf_token,
    vector_size=vector_dim,
    window=window_size,
    min_count=min_word_count,
    workers=num_workers,
    sg=sg 
)

print(f"Modelo treinado com vocabulário de {len(model_w2v_transcricao_token.wv.index_to_key)} palavras.")

In [ ]:
model_w2v_transcricao_token.wv.most_similar("votacao")

In [ ]:
vetores_documentos_transcricao = calcula_vetor_medio(corpus_tf_idf_token, model_w2v_transcricao_token, vector_dim)

In [ ]:
matriz_similaridade_top_n(vetores_documentos_transcricao, doc_ids_tf_idf_token, 15, "sumário Word2Vec - token")

In [ ]:
df_w2v_docs_transcricao = k_means(doc_ids_tf_idf_token, vetores_documentos_transcricao, num_clusters)

In [ ]:
mostrar_grafico_clusters(vetores_documentos_transcricao, df_w2v_docs_transcricao)

### Sumário

#### Tokens

In [23]:
corpus_sumario,doc_ids_sumario = get_discursos_corpus("sumario",CAMPOS_ID, QUANTITY)   
corpus_sumario_td_idf_token,doc_ids_sumario_tf_idf_token = get_discursos_corpus("sumario_tokens",CAMPOS_ID, QUANTITY)

##### TF-IDF

In [ ]:
x_sumario_tf_idf_token, vec_sumario_td_idf_token = processar_dados_td_idf(corpus_sumario_td_idf_token)

In [ ]:
mostrar_tabela(x_sumario_tf_idf_token,vec_sumario_td_idf_token,"Matriz Documento-Termo Sumario (TF-IDF)")

In [ ]:
search_and_rank(query, vec_sumario_td_idf_token, x_sumario_tf_idf_token, corpus_sumario_td_idf_token, doc_ids_sumario_tf_idf_token, "TF-IDF - sumario - tokens")

In [ ]:
matriz_similaridade_top_n(x_sumario_tf_idf_token, doc_ids_sumario_tf_idf_token, 15, "sumário - token")

In [ ]:
discurso_a = database.get_discurso_by_id(204553, 9)
discurso_b = database.get_discurso_by_id(109429, 6)

print(discurso_a["sumario"])
print('-'*50)
print(discurso_b["sumario"])

In [ ]:
grafico2D(x_sumario_tf_idf_token)

In [ ]:
grafico3D(x_sumario_tf_idf_token)

##### Word2Vec


In [ ]:
vector_dim = 100 # Dimensão dos vetores
window_size = 5  # Janela de contexto
min_word_count = 1 # Contagem mínima para incluir palavra
num_workers = 4 # Threads para treinamento
sg = 0 # Usando CBOW (padrão), 1 para Skip-gram

model_w2v_sumario_token = Word2Vec(
    sentences=corpus_sumario_td_idf_token,
    vector_size=vector_dim,
    window=window_size,
    min_count=min_word_count,
    workers=num_workers,
    sg=sg 
)

print(f"Modelo treinado com vocabulário de {len(model_w2v_sumario_token.wv.index_to_key)} palavras.")

In [ ]:
model_w2v_sumario_token.wv.most_similar("votacao")

In [ ]:
vetores_documentos = calcula_vetor_medio(corpus_sumario_td_idf_token,model_w2v_sumario_token,vector_dim)

In [ ]:
matriz_similaridade_top_n(vetores_documentos, doc_ids_sumario_tf_idf_token, 15, "sumário Word2Vec - token")

In [ ]:
recomendacoes_df = busca_termos_similares(model_w2v_sumario_token,doc_ids_sumario_tf_idf_token, vetores_documentos, 5)

recomendacoes_df

In [ ]:
df_w2v_docs = k_means(doc_ids_sumario_tf_idf_token,vetores_documentos,10)

In [ ]:
mostrar_grafico_clusters(vetores_documentos, df_w2v_docs)

#### BERT

In [24]:
embeds = gerar_embeds(corpus_sumario)

In [ ]:
resultados = []

for (modelo, pooling), X in embeds.items():
    # Similaridade (agora desempacotando corretamente)
    S, _ = plot_similarity_heatmap_px(
        X,
        sentences=corpus_sumario,  # <- inclui textos no hover
        title=f"Similaridade ({modelo}, {pooling})",
        cmap="Blues",
        mask_upper=False,     # ou True, se quiser só triângulo inferior
        vmin=0.0, vmax=1.0,
        cbar_label="Similaridade",
        fmt=".2f",
        xgap=1, ygap=1
    )
    # Clustering
    labels = run_kmeans(X, n_clusters=3, random_state=SEED)
    resultados.append(pd.DataFrame({
        "modelo": modelo,
        "pooling": pooling,
        "sentenca": corpus_sumario,
        "cluster": labels
    }))

clusters_df = pd.concat(resultados, ignore_index=True)

In [26]:
# --- 1) Preparar dados 2D (PCA ou UMAP) ---
df2d = plot_scatter_embeddings(embeds, sentences=corpus_sumario, method="pca")

# Corrige a chave de merge: é 'sentenca' (singular), não 'sentencas'
df2d = df2d.merge(clusters_df, on=["modelo", "pooling", "sentenca"], how="left")

# ID curto para cada sentença (S1, S2, …)
df2d["sid"] = df2d["sentenca"].apply(lambda s: f"S{corpus_sumario.index(s)+1}")

# --- 2) Converter 'cluster' para categórico (garante legenda discreta) ---
df2d["cluster"] = df2d["cluster"].astype(str)

# --- 3) Ordem explícita dos facetes ---
model_order = ["mBERT", "BERTimbau", "BERTugues"]
pool_order  = sorted(df2d["pooling"].unique())
cluster_order = sorted(df2d["cluster"].unique())

# --- 4) Paleta discreta para clusters (tons de azul) ---
k = int(df2d["cluster"].nunique())
blue_seq = px.colors.sequential.Blues
while len(blue_seq) < k:
    blue_seq = blue_seq + blue_seq
color_seq = blue_seq[-k:]

# --- 5) Scatter interativo com facetas e hover detalhado ---
fig = px.scatter(
    df2d,
    x="x", y="y",
    color="cluster",
    color_discrete_sequence=color_seq,
    facet_col="modelo",
    facet_row="pooling",
    facet_col_spacing=0.08,
    facet_row_spacing=0.10,
    category_orders={
        "modelo": model_order,
        "pooling": pool_order,
        "cluster": cluster_order
    },
    hover_data={
        "sid": True,
        "sentenca": True,
        "modelo": True,
        "pooling": True,
        "cluster": True,
        "x": ':.3f',
        "y": ':.3f'
    },
    title="Embeddings 2D por Modelo (colunas) e Pooling (linhas) — PCA"
)

fig.update_layout(
    template="plotly_white",
    legend_title_text="Cluster",
    margin=dict(l=40, r=20, t=60, b=40),
)

fig.update_traces(
    marker=dict(size=10, line=dict(width=0)),
    opacity=0.9,
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"     # sid
        "Sentença: %{customdata[1]}<br>"
        "Modelo: %{customdata[2]} | Pooling: %{customdata[3]}<br>"
        "Cluster: %{customdata[4]}<br>"
        "x: %{x:.3f} | y: %{y:.3f}<extra></extra>"
    )
)

# Molduras ao redor de cada faceta (opcional)
for xaxis_name in [k for k in fig.layout if k.startswith("xaxis")]:
    xdom = getattr(fig.layout, xaxis_name).domain
    suffix = xaxis_name[5:]  # "" ou "2","3",...
    yaxis_name = "yaxis" + suffix
    if hasattr(fig.layout, yaxis_name):
        ydom = getattr(fig.layout, yaxis_name).domain
        fig.add_shape(
            type="rect",
            xref="paper", yref="paper",
            x0=xdom[0], x1=xdom[1], y0=ydom[0], y1=ydom[1],
            line=dict(color="rgba(0,0,0,0.28)", width=1),
            fillcolor="rgba(0,0,0,0)",
            layer="below"
        )

fig.show()

NameError: name 'clusters_df' is not defined

#### Lematização

In [ ]:
corpus_sumario_td_idf_lem,doc_ids_sumario_tf_idf_lem = get_discursos_corpus("sumario_lemizado",CAMPOS_ID, QUANTITY)
x_sumario_tf_idf_lem, vec_sumario_td_idf_lem = processar_dados_td_idf(corpus_sumario_td_idf_lem)

In [ ]:
mostrar_tabela(x_sumario_tf_idf_lem,vec_sumario_td_idf_lem,"Matriz Documento-Termo Sumario (TF-IDF)")

In [ ]:
matriz_similaridade_top_n(x_sumario_tf_idf_lem, doc_ids_sumario_tf_idf_lem,15, "sumario - lematizado")

In [ ]:
search_and_rank(query, vec_sumario_td_idf_lem, x_sumario_tf_idf_lem, corpus_sumario_td_idf_lem, doc_ids_sumario_tf_idf_lem, "TF-IDF - sumario - lem")

In [ ]:
discurso_a = database.get_discurso_by_id(220573, 11)
discurso_b = database.get_discurso_by_id(74159, 9)

print(discurso_a["sumario"])
print('-'*50)
print(discurso_b["sumario"])